In [23]:
import numpy as np
import pandas as pd

In [24]:
df = pd.read_csv('Loan_approval_data_2025.csv').drop(columns=['customer_id','occupation_status','product_type','loan_intent'])

In [25]:
df.head()

,age,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,delinquencies_last_2yrs,derogatory_marks,loan_amount,interest_rate,debt_to_income_ratio,loan_to_income_ratio,payment_to_income_ratio,loan_status
0,40,17.2,25579,692,5.3,895,10820,0,0,0,600,17.02,0.423,0.023,0.008,1
1,33,7.3,43087,627,3.5,169,16550,0,1,0,53300,14.10,0.384,1.237,0.412,0
2,42,1.1,20840,689,8.4,17,7852,0,0,0,2100,18.33,0.377,0.101,0.034,1
3,53,0.5,29147,692,9.8,1480,11603,0,1,0,2900,18.74,0.398,0.099,0.033,1
4,32,12.5,63657,630,7.2,209,12424,0,0,0,99600,13.92,0.195,1.565,0.522,1


In [26]:
x = df.drop(columns=['loan_status'])
y = df['loan_status']

In [27]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [28]:
from sklearn.linear_model import LogisticRegression,LinearRegression
lr = LinearRegression()
lr.fit(x_train,y_train)
y_pred = lr.predict(x_test)
from sklearn.metrics import r2_score
r2_score(y_test,y_pred)

0.4756170439358848

In [54]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
lda = LDA(n_components=1)
x_train_lda = lda.fit_transform(x_train,y_train)
x_test_lda = lda.transform(x_test)

In [55]:
from sklearn.linear_model import LogisticRegression,LinearRegression
lr = LinearRegression()
lr.fit(x_train_lda,y_train)
y_pred = lr.predict(x_test_lda)
from sklearn.metrics import r2_score
r2_score(y_test,y_pred)

0.47561704393577486

In [48]:
class Own_LDA:

    def __init__(self,n_components):
        self.m = n_components


    def fit_transform(self,x_train,y_train):
        # Get the number of feature
        self.n_feature = x_train.shape[1]

        # Get The unique class labels
        class_label = np.unique(y_train)

        # Get the overall mean vector
        overall_mean = np.mean(x_train, axis=0)

        # Initialize both scatter matices to zeros
        sw = np.zeros((self.n_feature, self.n_feature)) # within scatter matix
        sb = np.zeros((self.n_feature, self.n_feature)) # between scatter matix

        # Iterative the over all classes and select corresponding data
        for c in class_label:
            # Filter the x_train for y_train class
            xc = x_train[y_train == c]
            # calculate the mean vector for class c
            mean_c = np.mean(xc, axis=0)
            # Calculate within scatter matrix
            sw += (xc - mean_c).T.dot((xc - mean_c))
            # Number of samples in class c
            n_c = xc.shape[0]
            # Difference between the overall mean and the mean of class c --> between-class scatter
            mean_diff = (mean_c - overall_mean).values.reshape(self.n_feature, 1)
            sb += n_c * (mean_diff).dot(mean_diff.T)

        # Determine SW^-1 * SB
        A = np.linalg.inv(sw).dot(sb)
        # Get the eigenvalues and eigenvectors of (SW^-1 * SB)
        eigenvalues, eigenvectors = np.linalg.eig(A)
        # Keep only the real parts of eigenvalues and eigenvectors
        eigenvalues = np.real(eigenvalues)
        eigenvectors = np.real(eigenvectors.T)


         # Sort the eigenvalues descending (high to low)
        idxs = np.argsort(np.abs(eigenvalues))[::-1]
        self.eigenvalues = np.abs(eigenvalues[idxs])
        self.eigenvectors = eigenvectors[idxs]
        # Store the first m eigenvectors as transformation matrix W
        self.w = self.eigenvectors[0:self.m]

        # Transform the feature matrix X onto LD axes
        return np.dot(x_train, self.w.T)


    def transform(self,x_test):
        return np.dot(x_test, self.w.T)
        
            
    

In [49]:
lda = Own_LDA(n_components=1)
x_train_lda2 = lda.fit_transform(x_train,y_train)
x_test_lda2 = lda.transform(x_test)

In [50]:
from sklearn.linear_model import LogisticRegression,LinearRegression
lr = LinearRegression()
lr.fit(x_train_lda2,y_train)
y_pred = lr.predict(x_test_lda2)
from sklearn.metrics import r2_score
r2_score(y_test,y_pred)

0.47561704393850845